# Embedding movie descriptions
This project creates embeddings of movie descriptions using pretrained model. Those embeddings are used to implement RAG-like functionality. Using UMAP algorithm I mapped those embeddings to 2D space for visualization. I used HDBSCAN to find clusters automatically. 

In [1]:
import torch
from utils import load_device
import numpy as np

In [2]:
device=load_device()

PyTorch Version: 2.8.0+cu126
Using GPU: NVIDIA GeForce GTX 1660 Ti


In [3]:
import pandas as pd 
df=pd.read_csv('datasets/wiki_movie_plots_deduped.csv')

In [4]:
df.drop_duplicates(subset=['Plot'],inplace=True)

In [5]:
descriptions = df['Plot'].values

In [6]:
from sentence_transformers import SentenceTransformer
#sbert = SentenceTransformer("all-MiniLM-L6-v2")
st=SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")

In [7]:
from tqdm.notebook import tqdm
tqdm.pandas()

st.half()

embeddings = st.encode(
    descriptions,
    convert_to_tensor=True,
    batch_size=256,
    normalize_embeddings=True
)

In [22]:
torch.save(embeddings, 'movie_embeddings.pt')

In [ ]:
# To load it later:
# loaded_embeddings = torch.load('movie_embeddings.pt')

In [8]:
def find_movie(query,embeddings,movies,top_k=5): 
    query_embed=st.encode(query, convert_to_tensor=True)
    similarities=st.similarity(query_embed,embeddings).squeeze(dim=0).cpu()
    indices=torch.argsort(similarities,descending=True)[:top_k]
    return movies.iloc[indices]

In [20]:
found_movies=find_movie("Cars racing in japan",embeddings,df)

In [21]:
found_movies

,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot,clean_genre,main_genre,plot_genre
23554,2005,Initial D,Hong Kong,"Alan Mak, Andrew Lau","Jay Chou, Shawn Yue, Edison Chen, Anthony Wong...",unknown,https://en.wikipedia.org/wiki/Initial_D_(film),"Taking place in Gunma prefecture in Japan, the...",unknown,other,other
14907,2006,The Fast and the Furious: Tokyo Drift,American,Justin Lin,"Lucas Black, Bow Wow",action,https://en.wikipedia.org/wiki/The_Fast_and_the...,"In 2015, Oro Valley, Arizona, high school stud...",action,action,action
15994,2011,Cars 2,American,"John Lasseter, Brad Lewis (co-director)","Owen Wilson, Larry the Cable Guy, Michael Cain...","animated, family, spy film",https://en.wikipedia.org/wiki/Cars_2,"Finn McMissile, a British spy, infiltrates the...","animated, family, spy film",animated,other
33783,2014,New Initial D the Movie,Japanese,Masamitsu Hidaka,NaN,unknown,https://en.wikipedia.org/wiki/New_Initial_D_th...,The movie opens with Koichiro Iketani driving ...,unknown,other,other
23396,1995,Full Throttle,Hong Kong,Derek Yee,"Andy Lau, Gigi Leung, David Wu, Paul Chun, Chi...",drama,https://en.wikipedia.org/wiki/Full_Throttle_(f...,Joe (Andy Lau) is a youth that loves auto raci...,drama,drama,drama


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
model_name = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="auto")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the cpu.


In [12]:
def generate_answer(user_query,model,tokenizer,embeddings,df,top_k=3):
    found_movies=find_movie(user_query,embeddings,df,top_k=top_k)
    plots=found_movies['Plot'].values.tolist()
    titles=found_movies['Title'].values.tolist()
    retrieved_contexts=[f"Title: {title}\nPlot: {plot}"for title,plot in zip(titles,plots)]
    combined_context="\n\n---\n\n".join(retrieved_contexts)
    system_prompt = (
        "You are a knowledgeable movie expert chatbot. You will be provided with several retrieved movie plots. "
        "Review the plots and suggest the most relevant one(s) based on the user's request. "
        "Follow these STRICT formatting rules:\n"
        "1. Never use markdown bolding (**) for movie titles. Always enclose movie titles in double quotation marks (e.g., \"Two-Lane Blacktop\").\n"
        "2. Never refer to internal context labels like 'Option 1', 'Context 2', or 'Document 3'. Present the suggestions naturally.\n"
        "3. Never mention movies that strictly match the user's criteria (e.g., if they ask for car racing, do not recommend boat racing movies)."
    )
    user_message = f"Available Contexts (Movie Plots):\n{combined_context}\n\nUser Question: {user_query}"
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_message}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=512,
    )
    generated_ids = [
        output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
    ]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return response

In [13]:
user_query="Suggest good car racing movies"
response=generate_answer(user_query,model,tokenizer,embeddings,df)
print(response)

Based on your request for good car racing movies, I would recommend the following:

Option 1: "The Gumball Rally"
This movie features a unique and exciting road rally across the United States, focusing on the camaraderie and competition among various teams. It includes elements of humor and unexpected twists, making it a thrilling watch for car racing enthusiasts.

Option 2: "Red Line 7000"
While this movie does not focus entirely on car racing, it offers a compelling story involving a racing team and a 1965 Shelby GT-350 race car. The detailed attention to period-specific vehicles and the dramatic racing scenes make it a worthy addition to any car racing film collection.


In [14]:
df['clean_genre'] = df['Genre'].astype(str).str.lower().str.strip()
df['main_genre']=df['clean_genre'].str.split(r'[ ,/|-]').str[0]
genre_mapping = {
    'sci': 'science fiction',
    'sci-fi': 'science fiction',
    'sci fi': 'science fiction',
    'rom com': 'romance',
    'romantic': 'romance',
    'romantic comedy': 'romance',
    'action comedy': 'action',
    'unknown': 'other'
}
df['main_genre'] = df['main_genre'].replace(genre_mapping)

In [15]:
top_n=12
top_genres=df['main_genre'].value_counts().index[:top_n].tolist()

In [16]:
df['plot_genre']=df['main_genre'].apply(lambda x:x if x in top_genres else 'other')
mask=df['plot_genre'] != 'other'
df_filtered = df[mask].copy()

In [17]:
import umap

reducer = umap.UMAP(
    n_neighbors=30,    # Balances local vs global structure
    min_dist=0.01,      # Controls how tightly points are packed
    metric='cosine',   # Cosine distance works much better for text embeddings than Euclidean
    n_components=2,    # Reduce to 2D for a scatter plot
    random_state=42    # For reproducibility
)
embeddings_2d = reducer.fit_transform(embeddings.cpu().numpy()[mask])
df_filtered['x'] = embeddings_2d[:, 0]
df_filtered['y'] = embeddings_2d[:, 1]

C:\Users\barte\hands_on_ml\.venv\Lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [18]:
import plotly.express as px
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=20, 
    min_samples=10,      
    gen_min_span_tree=False
)
cluster_labels = clusterer.fit_predict(embeddings_2d)
df_filtered['cluster'] = [f"Cluster {label}" if label != -1 else "Noise (-1)" for label in cluster_labels]
fig = px.scatter(
    df_filtered,
    x='x',
    y='y',
    color='cluster',
    hover_name='Title',
    hover_data={'x': False, 'y': False, 'Genre': True,'cluster':False,'Release Year': True,'Origin/Ethnicity':True},
    title="Wikipedia Movie Plots: SBERT Embeddings Clustered by Genre",
    render_mode='webgl'
)

fig.update_traces(marker=dict(size=4, opacity=0.6, line=dict(width=0)))

for trace in fig.data:
    if trace.name == "Noise (-1)":
        trace.marker.color = "lightgray"
        trace.marker.opacity = 0.3

fig.update_layout(
    xaxis=dict(showgrid=False, zeroline=False, visible=False),
    yaxis=dict(showgrid=False, zeroline=False, visible=False),
    template="plotly_white",
    margin=dict(l=20, r=20, t=50, b=20)
)

fig.show(renderer="iframe")